# Sanity check Top-N: Popularity, MF, TF-IDF

Mengecek bahwa tiga baseline menghasilkan top-10 yang masuk akal pada subset kecil validation (200 user, seed 42). Ini bukan evaluasi resmi (itu W10). Semua model hanya dilatih dari train.

In [1]:
import sys, random, datetime
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data_loader import load_data, relevant_items_val
from src.popularity import PopularityRecommender
from src.matrix_factorization import MatrixFactorizationRecommender
from src.content_based import TfidfContentRecommender

K, N_USERS, SEED = 10, 200, 42

D:\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Muat data

Data train, validation, kandidat warm, dan item yang sudah dilihat user di train. `items.csv` dipakai untuk menampilkan judul produk, dan TF-IDF memakai kolom `content_text_basic` dari file yang sama sebagai fitur.

In [2]:
train, val, cand_set, seen = load_data(str(ROOT / 'data') + '/')
rel = relevant_items_val(val, cand_set, seen)
print(f'train={len(train):,}; validation={len(val):,}; candidates={len(cand_set):,}; evaluable users={len(rel):,}')

items = pd.read_csv(ROOT / 'data' / 'items.csv', encoding='utf-8-sig', usecols=['item_id', 'title'])
title = dict(zip(items['item_id'], items['title']))

train=427,957; validation=41,284; candidates=22,753; evaluable users=11,850


## Latih tiga baseline

Parameter sama dengan notebook demo masing-masing

### Popularity

In [3]:
pop = PopularityRecommender().fit(train, cand_set)
print('Popularity siap:', len(pop.ranked_items), 'item kandidat berperingkat')

Popularity siap: 22672 item kandidat berperingkat


### MF

In [4]:
mf = MatrixFactorizationRecommender(
    n_factors=32, n_epochs=10, learning_rate=0.01,
    regularization=0.02, min_rating=4, random_state=42,
).fit(train, cand_set)
print('MF siap:', len(mf.user_index), 'user,', len(mf.item_index), 'item')

MF siap: 54483 user, 22753 item


### TF-IDF

In [5]:
tfidf = TfidfContentRecommender().fit(train, cand_set)
print('TF-IDF siap:', len(tfidf.item_index), 'item')

TF-IDF siap: 22753 item


## Subset user validation

Ambil 200 user acak dari evaluable users dengan seed tetap, supaya hasilnya bisa diulang.

In [6]:
recommenders = {
    'Popularity': lambda u: pop.recommend(u, seen, K),
    'MF (Chris)': lambda u: mf.recommend(u, seen, K),
    'TF-IDF CBF (Tiffani)': lambda u: tfidf.recommend(u, seen, K),
}
random.seed(SEED)
users = random.sample(sorted(rel), N_USERS)
print(f'subset: {len(users)} user dari {len(rel):,} evaluable users (seed={SEED})')

subset: 200 user dari 11,850 evaluable users (seed=42)


## Cek aturan

Untuk tiap user dan tiap model: (1) tepat 10 item, (2) tidak ada item ganda, (3) tidak ada item yang sudah dilihat user di train, (4) semua item termasuk kandidat warm. Recall@10 di sini hanya angka kasar pada 200 user, bukan hasil resmi.

In [7]:
rows = []
for name, rec_fn in recommenders.items():
    problems, recalls = 0, []
    for u in users:
        recs = rec_fn(u)
        ok = (len(recs) == K and len(set(recs)) == K
              and not (set(recs) & seen[u]) and set(recs) <= cand_set)
        problems += (not ok)
        recalls.append(len(set(recs) & rel[u]) / len(rel[u]))
    rows.append({'Model': name, 'User bermasalah': f'{problems}/{N_USERS}',
                 'Recall@10 kasar': round(float(np.mean(recalls)), 4)})
summary = pd.DataFrame(rows)
summary

,Model,User bermasalah,Recall@10 kasar
0,Popularity,0/200,0.0108
1,MF (Chris),0/200,0.0000
2,TF-IDF CBF (Tiffani),0/200,0.0133


## Contoh rekomendasi

Top-10 satu user yang sama untuk tiga model, lengkap dengan judul produk, untuk dicek apakah masuk akal.

In [8]:
u0 = users[0]
print('contoh user:', u0)
for name, rec_fn in recommenders.items():
    print(f'\n[{name}]')
    for i, it in enumerate(rec_fn(u0), 1):
        print(f'  {i}. {it} | {str(title.get(it, "?"))[:70]}')

contoh user: AHKXVSFVRVTI3DMAWLTANLZB5H4A

[Popularity]
  1. B09857JRP2 | GLS Audio Instrument Cable - Amp Cord for Bass & Electric Guitar - Str
  2. B0BCK6L7S5 | Donner Guitar Delay Pedal, Yellow Fall Analog Delay Guitar Effect Peda
  3. B0BSGM6CQ9 | Ernie Ball Mondo Slinky Nickelwound Electric Guitar Strings 10.5-52 Ga
  4. B0BPJ4Q6FJ | D'Addario Guitar Strings - XL Nickel Electric Guitar Strings - EXL150H
  5. B0BTC9YJ2W | D'Addario Guitar Strings - Phosphor Bronze Acoustic Guitar Strings - E
  6. B004XNK7AI | Snark SN-5 Tuner for Guitar, Bass and Violin (Black)
  7. B0BHSG5C3G | Snark SN5X Clip-On Tuner for Guitar, Bass & Violin (Current Model) 1.8
  8. B08R5GM6YB | D'Addario Pro-Winder - Guitar String Winder, Guitar String Cutter, Gui
  9. B0C67HCGBR | GLS Audio 12ft Mic Cable Patch Cords - XLR Male to XLR Female Black Ca
  10. B0BPLFP5P6 | D'Addario Accessories Guitar Tuner - Micro Headstock Tuner - Tuner for

[MF (Chris)]
  1. B003JCF5Y2 | GLS Audio 25ft Colored Blue Mic Cable P

## Simpan log

Log dibuat otomatis dari output di atas (tidak diedit manual), disimpan di `results/sanity_check_log.txt`.

In [9]:
(ROOT / 'results').mkdir(exist_ok=True)
lines = [f'Sanity check Top-N | {datetime.date.today()} | seed={SEED} | users={N_USERS} | K={K}',
         f'evaluable users (validation, warm): {len(rel)}', '', summary.to_string(index=False), '']
for name, rec_fn in recommenders.items():
    lines.append(f'[{name}] contoh user {u0}:')
    for i, it in enumerate(rec_fn(u0), 1):
        lines.append(f'   {i}. {it} | {str(title.get(it, "?"))[:70]}')
    lines.append('')
(ROOT / 'results' / 'sanity_check_log.txt').write_text('\n'.join(lines), encoding='utf-8')
print('log tersimpan di results/sanity_check_log.txt')

log tersimpan di results/sanity_check_log.txt
